In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, LabelEncoder, StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix



In [ ]:
url = 'https://raw.githubusercontent.com/niksonlaurentino/SCTECH_Projeto_Avaliativo_M_1/fase/eda/dataset/credit_risk_dataset.csv'

In [ ]:
df = pd.read_csv(url)

# Fase 1: Análise Exploratória de Dados (EDA)
# Descritiva e Estatística: tamanho total da base (linhas e colunas)

In [ ]:
df.shape

# Tipos de dados de cada variável 

In [ ]:
df.info()

# Sumário estatístico descritivo (médias, mínimos, máximos e quartis via .describe())

In [ ]:
df.describe()

# Frequência de aparecimento de valores CATEGÓRICOS e NUMÉRICOS

In [ ]:
cat_cols = df.select_dtypes(include='str').columns
num_cols = df.select_dtypes(include='number').columns

In [ ]:
print(f'{50*'='} COLUNAS CATEGORICAS {50*'='}')
for column in cat_cols:
    print(f'COLUNA: {df[column].value_counts()}')
    print(f'{50*'-'}')

print(f'{50*'='} COLUNAS NUMERICAS {50*'='}')
for column in num_cols:
    print(f'COLUNA: {df[column].value_counts()}')
    print(f'{50*'-'}')

# Análise Visual: 
# Histograma de distribuição de idades/valores

In [ ]:
sns.set_theme(style='whitegrid')
plt.figure(figsize=[10,8])
sns.histplot(df, x='person_age')
plt.ylabel('Valores')
plt.xlabel('Idades')
plt.show()

# Gráfico de barras provando o desbalanceamento da variável alvo 

In [ ]:
plt.figure(figsize=[10, 8])
sns.countplot(data=df, x='loan_status')
plt.title('Distribuição de Loan Status')
plt.xlabel('Status do Empréstimo')
plt.ylabel('Contagem')
plt.show()

# Mapa de calor de correlação de Pearson das variáveis numéricas

In [ ]:
num_cols = df.select_dtypes(include='number')
plt.figure(figsize=[10,10])
sns.heatmap(num_cols.corr(), annot=True, fmt='.1f', linewidth=.5)
plt.title('Matriz de Correlação de Pearson para elementos numéricos')
plt.show()

In [ ]:
plt.figure(figsize=[10,10])
sns.heatmap(num_cols.corr('spearman'), annot=True, fmt='.1f', linewidth=.5,  cmap="crest")
plt.title('Matriz de Correlação de Spearman para elementos numéricos')
plt.show()

# TOMADA DE DECISÃO
O dataframe é composto por 32.581 linhas e 12 colunas, das quais 8 são numéricas e 4 categóricas. Através do método describe(), constatamos que a variável numérica idade (person_age) não apresenta valores mínimos negativos, o que confirma a consistência dos dados. Observa-se também uma forte disparidade entre a média e a mediana na variável de rendimento (person_income), indicando a presença de diversos outliers. Quanto ao mapa de calor com as correlações de Pearson e Spearman, que medem a associação entre as variáveis, vale destacar que coeficientes acima de 0,75 ou abaixo de -0,75 devem ser analisados com cautela para evitar multicolinearidade. Como a variável person_age possui uma correlação alta de 0,90 com cb_person_cred_hist_length, uma abordagem recomendada é a remoção de uma delas quando o algoritmo for KNN, mas se o modelo for random forest não será preciso excluir essa coluna, tão pouco padronizar os valores numéricos.
* OBS: Faremos a imputação por mediana após o split para evitar vazamento de dados (data leakage).

# Fase 2: Tratamento e Limpeza (Data Prep)

# Dropando as linhas duplicadas

In [ ]:
df = df.drop_duplicates()

# Valores Nulos

Podemos constatar a presença de nulos para as colunas da seguinte forma:
* person_emp_length             2.74% de nulos com um total de 887
* loan_int_rate                 9.55% de nulos com um total de 3095

Baseado no estudo anterior, podemos fazer a imputação estatística por mediana, pois ela é menos suscetível a presença de outliers, pois um dos algoritmos que usaremos será o KNN que é sensível a distâncias.


In [ ]:
round((df.isnull().sum() / df.shape[0]) * 100,2), df.isnull().sum()

# Tratamento de Outliers
# Boxplot para Evidenciar Outliers
Nota-se que após o bigode superior temos várias ocorrências de valores. O limite do bigode superior é calculado pela fórmula estatística Q3 + 1,5 x (IQR).

In [ ]:
plt.figure(figsize=[30,6])
sns.boxplot(data=df[['person_age', 'person_income', 'person_emp_length', 'loan_amnt',
       'loan_int_rate', 'loan_percent_income',
       'cb_person_cred_hist_length']], orient='h')
plt.xscale('log') #a escala tem que ser em logarítmico, pois senão os valores ficam empilhados
plt.title('Distribuição de Frequências')
plt.xlabel('Escala Logarítmica')
plt.show()

# Análise de Outliers
Vemos em quase todas as variáveis a presença massiva de outliers. Como vamos trabalhar com dois algoritmos, vamos usar um dataframe voltado para KNN removendo outliers e outro dataframe sem a remoção de outliers para Random Forest. 

# Fase 3: Feature Engineering (Coluna Calculada)
Como as colunas loan_amnt e person_income não tem valores nulos como vimos anteriormente, podemos utilizá-las para criar uma nova coluna sem problemas.

In [ ]:
df['comprometimento_renda']  = round((df['loan_amnt'] / df['person_income']) * 100,2) # coloquei o round para ficar apenas dois números após a vírgula
#Montamos duas copias de dataset para tratamentos dedicados para cada algoritmo
df_pre_processing = df.copy()

* O df_pre_processing servirá de bifurcação para iniciarmos dois tratamentos distintos: um para KNN e outro para Random Forest

# Fase 4: Separação, Balanceamento e Escalonamento Seguro

In [ ]:
y = df_pre_processing['loan_status']
x = df_pre_processing.drop(columns=['loan_status'])
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42, stratify=y
)


# IMPUTAÇÃO PELA MEDIANA (Evita vazamento de dados)
Aqui temos a bifurcação de processos, ou seja, para evitar repetição desnecessária no código usaremos a imputação por mediana também para random forest

In [ ]:
num_cols_preprocessing = x_train.select_dtypes(include='number').columns

imputer = SimpleImputer(strategy='median')
x_train[num_cols_preprocessing] = imputer.fit_transform(x_train[num_cols_preprocessing])
x_test[num_cols_preprocessing] = imputer.transform(x_test[num_cols_preprocessing])

# BIFURCAR utilizar o x_train e x_test para RANDOM FOREST APÓS O PIPELINE KNN

# CORTE DE OUTLIERS PARA KNN
Usaremos a amplitude interquartil como referência para limites superiores e inferiores para exclusão de outliers.

In [ ]:
x_train_knn = x_train.copy()
num_cols = x_train_knn.select_dtypes(include='number').columns

for col in num_cols:
    Q1 = x_train_knn[col].quantile(0.25) 
    Q3 = x_train_knn[col].quantile(0.75)
    IQR = Q3 - Q1
    
    limite_inferior = Q1 - 1.5 * IQR
    limite_superior = Q3 + 1.5 * IQR
    
    # Aplica o corte direto acumulado
    x_train_knn = x_train_knn[(x_train_knn[col] >= limite_inferior)& (x_train_knn[col] <= limite_superior)]

# Sincroniza o y_train com os índices mantidos
y_train_knn = y_train.loc[x_train_knn.index]


# Utilização de Ordinal Encoder para mapear a variável loan_grade por "Peso de Nota"
* OBSERVAÇÃO: Variáveis Categóricas do Tipo Ordinais tem relação de hierarquia entre a cardinalidade, ou seja, a coluna loan_grade (Nota de risco do empréstimo) possui notas de avaliação de crédito: A para melhor e G para pior avaliação. Sendo assim, não podemos apenas "binarizar" as colunas. Precisamos evidenciar o peso da variação da nota para nosso target. Porém após usarmos o Ordinal Encoder com mapeamento manual, vamos padronizar também essa coluna, pois o KNN é sensível a distâncias não padronizadas. 
* Nesse primeiro momento montamos o pipeline de pré-execução.



In [ ]:
ordem_grade = [['A', 'B', 'C', 'D', 'E', 'F', 'G']] # mapeamento manual

pipeline_ordinal = Pipeline(steps=[
    ('encoder', OrdinalEncoder(categories=ordem_grade))
    # (#'scaler', StandardScaler() 
    #     ) # segundo o enunciado seria para aplicar a padronização SOMENTE para variáveis contínuas
])



Separamos os nomes das colunas Categóricas das Numéricas para tratamento de Encode. Utilizamos o ColumnTransformer para executar a Padronização, o Ordinal Encoder e Padronização do elemento Loan_Grade e por fim o One-Hot-Encoder para elementos Categóricos.

In [ ]:
cat_cols_preprocessing = x_train.select_dtypes(include=['object', 'category', 'str']).columns.drop('loan_grade', errors='ignore')
num_cols_preprocessing = x_train.select_dtypes(include='number').columns

preprocessor = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', num_cols_preprocessing),             # Padroniza as numéricas puras
        ('ord', pipeline_ordinal, ['loan_grade']),       # Executa ENCODER -> SCALER no loan_grade
        ('nom', OneHotEncoder(handle_unknown='ignore'), cat_cols_preprocessing) # Aplica OHE nas nominais
    ]
)

scaler_pos_smote = ColumnTransformer(
    transformers=[('scaler', StandardScaler(), num_cols_preprocessing)], 
    remainder='passthrough'  # Mantém loan_grade e as colunas do OHE intactas
)

# 4. PIPELINE UNIFICADO (Roda tudo em uma única instrução)
pipeline_knn = Pipeline(steps=[
    ('encoder', preprocessor),   # Passo 1: Transforma textos em números
    ('smote', SMOTE(random_state=42)),   # Passo 2: Balanceia a matriz numérica 
    ('scaler', scaler_pos_smote),   
    ('knn', KNeighborsClassifier())      # Passo 3: Modelo final
])
# a linha de script ('smote', SMOTE(random_state=42)), substitui a insercao manual de:
# smote = SMOTE(random_state=42)
# x_train_resampled, y_train_resampled = smote.fit_resample(
#     x_train_scaled, y_train_knn
# )

# Escalonando com Padronização 

In [ ]:
# x_train_scaled = preprocessor.fit_transform(x_train_knn)
# x_test_scaled = preprocessor.transform(x_test)

pipeline_knn.fit(x_train_knn, y_train_knn)
y_pred = pipeline_knn.predict(x_test)

In [ ]:
# 2. Exibir o Relatório de Classificação (Precision, Recall, F1-Score)
print("--- RELATÓRIO DE CLASSIFICAÇÃO (KNN) ---")
print(classification_report(y_test, y_pred))

# 3. Exibir a Matriz de Confusão para ver os erros/acertos por classe
print("--- MATRIZ DE CONFUSÃO ---")
print(confusion_matrix(y_test, y_pred))

# KNN COM K3

In [ ]:
# knn = KNeighborsClassifier(n_neighbors=3)
# knn.fit(x_train_resampled, y_train_resampled)
# y_pred = knn.predict(x_test_scaled)